In [0]:
# ==================================================
# GOLD — Perfil das Vítimas
# Pipeline PRF Acidentes de Trânsito
# ==================================================

from pyspark.sql.functions import col, sum, count, round, desc

# Lê da Silver
df = spark.table("workspace.default.silver_acidentes")

# Agrega por classificação e fase do dia
df_gold_perfil = (df
    .groupBy("classificacao_acidente", "fase_dia", "condicao_metereologica")
    .agg(
        count("id").alias("total_acidentes"),
        sum("mortos").alias("total_mortos"),
        sum("feridos_graves").alias("total_feridos_graves"),
        sum("feridos_leves").alias("total_feridos_leves"),
        sum("ilesos").alias("total_ilesos"),
        sum("pessoas").alias("total_pessoas_envolvidas")
    )
    .withColumn("letalidade",
        round(col("total_mortos") / col("total_acidentes"), 4))
    .orderBy(desc("total_acidentes"))
)

# Grava Gold
(df_gold_perfil.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_perfil_vitimas")
)

print("Gold perfil vítimas gravada!")
df_gold_perfil.show(10, truncate=False)